In [0]:
df = spark.read.table("teams.sensorx.df_sx_800")

do horizon for the failiures - create a new column with failiure horizon based on some horizon H


Altering data for models:

Setting up data with lag and horizon

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# --- 1) Read data (all devices) ---
df_lr = spark.read.table("teams.sensorx.df_sx_800_with_delta")

# --- 2) One-Hot Encode properties_deviceId ---
indexer = StringIndexer(inputCol="properties_deviceId", outputCol="deviceId_index")
df_indexed = indexer.fit(df_lr).transform(df_lr)

encoder = OneHotEncoder(inputCol="deviceId_index", outputCol="deviceId_ohe")
df_encoded = encoder.fit(df_indexed).transform(df_indexed)

# --- 2b) Create failure horizon column (time-based) ---
# For each row at time t, failure_horizon = 1 if payload_fault_state == 1
# at any point from time t to time t + H_days (looking H_days into the future).
H_days = 10  # horizon in days
H_seconds = H_days * 86400  # convert to seconds

w_horizon = (
    Window.partitionBy("properties_deviceId")
    .orderBy(F.col("timeStamp").cast("long"))  # order by epoch seconds
    .rangeBetween(0, H_seconds)                 # look ahead H_seconds
)
df_encoded = df_encoded.withColumn(
    "failure_horizon",
    F.max(F.col("payload_fault_state").cast("int")).over(w_horizon)
)

# --- 3) Create lag features (partitioned by device for correctness) ---
sensor_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
]
n_lags = 20
w = Window.partitionBy("properties_deviceId").orderBy("timeStamp")

for L in range(1, n_lags + 1):
    for col_name in sensor_cols:
        df_encoded = df_encoded.withColumn(f"{col_name}{L}", F.lag(col_name, L).over(w))

lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]

# Drop rows with null lags (first n_lags rows per device)
df_clean = df_encoded.na.drop(subset=lag_cols)

# --- 4) Assemble features: sensor cols + OHE vector + lag cols ---
feature_input_cols = sensor_cols + ["deviceId_ohe"] + lag_cols
assembler = VectorAssembler(inputCols=feature_input_cols, outputCol="features")
df_features = assembler.transform(df_clean) \
    .select("timeStamp", "features", F.col("failure_horizon").alias("label"))

# --- 5) Chronological 80/20 train/test split ---
cutoff_epoch = df_features.selectExpr(
    "percentile_approx(CAST(timeStamp AS BIGINT), 0.8)"
).first()[0]

train = df_features.filter(F.col("timeStamp").cast("bigint") <= cutoff_epoch).cache()
test  = df_features.filter(F.col("timeStamp").cast("bigint") >  cutoff_epoch).cache()

Logistic Regression training:

In [0]:

# --- 6) Train Logistic Regression ---
lr = LogisticRegression(maxIter=100, regParam=0.0, elasticNetParam=0.0)
LRmodel = lr.fit(train)
LRpredictions = LRmodel.transform(test)

# --- 7) Evaluate ---
auc_eval = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
LRtrainingSummary = LRmodel.summary
LR_auc = auc_eval.evaluate(LRpredictions)
print(f"\nLogistic Regression with OHE + {n_lags} lags")
print(f"AUC: {LR_auc:.4f}")

# Confusion matrix
LRpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

Random forest with lag and horizon:

In [0]:
from pyspark.ml.classification import RandomForestClassifier

# --- RandomForest ---
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    seed=42
)

RFmodel = rf.fit(train)
RFpredictions = RFmodel.transform(test)

# Extract the summary from the returned RandomForestClassifier instance trained in the earlier example
RFtrainingSummary = RFmodel.summary

RF_auc = auc_eval.evaluate(RFpredictions)
print(f"\nRandom Forest with {n_lags} lags")
print(f"AUC: {RF_auc:.4f}")


# Confusion matrix
RFpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()


Gradient Boosted trees with lag and horizon:


In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.feature import StringIndexer, VectorIndexer

# Index labels, adding metadata to the label column.
labelIndexer = StringIndexer(inputCol="label", outputCol="indexedLabel").fit(train)

# Automatically identify categorical features, and index them.
featureIndexer = VectorIndexer(inputCol="features", outputCol="indexedFeatures", maxCategories=4, handleInvalid='skip').fit(train)

# Train a GBT model.
gbt = GBTClassifier(labelCol="indexedLabel", featuresCol="indexedFeatures", maxIter=10)

# Chain indexers and GBT in a Pipeline
pipeline = Pipeline(stages=[labelIndexer, featureIndexer, gbt])

# Train model.  This also runs the indexers.
model = pipeline.fit(train)

# Make predictions.
gbtpredictions = model.transform(test)

# --- Evaluate (same format as LR / RF) ---
GBT_auc = auc_eval.evaluate(gbtpredictions)
print(f"\nGradient Boosted Trees with {n_lags} lags")
print(f"AUC: {GBT_auc:.4f}")

# Confusion matrix
gbtpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

In [0]:
# Adding delta t to the table to see how much time has passed between measurements

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read table
df = spark.read.table("teams.sensorx.df_sx_800")

# Ordering by timestamp
order_cols = [F.col("timeStamp").asc()]
if  "serialNumber" and ("serialNumber" in df.columns):
    order_cols.append(F.col("serialNumber").asc())
w = Window.partitionBy("properties_deviceId").orderBy(*order_cols)

# Compute delta column in seconds. If negative then change to NULL
df_with_delta = (
    df
    .withColumn("prev_ts", F.lag("timeStamp").over(w))
    .withColumn(
        "delta_seconds",
        F.when(F.col("prev_ts").isNull(), F.lit(None).cast("double"))
         .otherwise(
             F.when(
                 (F.col("timeStamp").cast("long") - F.col("prev_ts").cast("long")) < 0,
                 None
             ).otherwise(F.col("timeStamp").cast("long") - F.col("prev_ts").cast("long"))
         )
    )
    .drop("prev_ts")
)


df_delta = spark.read.table("teams.sensorx.df_sx_800_with_delta")

display(df_delta.limit(30))